# Laboratorio: Transformación de texto en embeddings con TensorFlow

**Objetivo:** comprender cómo un texto se transforma en representaciones numéricas (*embeddings*) y aplicar estas representaciones en un modelo de clasificación de texto.

En este notebook se realizan:
1. Preparación y exploración de datos.
2. Tokenización y creación de secuencias.
3. Construcción de una capa `Embedding`.
4. Entrenamiento de una red neuronal.
5. Evaluación mediante precisión y pérdida.
6. Predicciones sobre textos.
7. Análisis de resultados.

> Este notebook está diseñado para ejecutarse en Google Colab.


In [ ]:
# 1. IMPORTACIÓN DE LIBRERÍAS
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("Versión de TensorFlow:", tf.__version__)


## 2. Dataset

Se utiliza el conjunto de datos **IMDB**, incluido en `tf.keras.datasets`. Contiene reseñas de películas etiquetadas como:
- `0`: sentimiento negativo
- `1`: sentimiento positivo

Para facilitar el experimento, se limita el vocabulario a 10.000 palabras y cada reseña se lleva a una longitud máxima de 200 tokens.


In [ ]:
# 2. CARGA DEL DATASET IMDB
VOCAB_SIZE = 10000
MAX_LEN = 200

(train_data, train_labels), (test_data, test_labels) = tf.keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

print("Ejemplos de entrenamiento:", len(train_data))
print("Ejemplos de prueba:", len(test_data))
print("Primera secuencia:", train_data[0][:20])
print("Primera etiqueta:", train_labels[0])


In [ ]:
# 3. AJUSTE DE LONGITUD
train_data = tf.keras.utils.pad_sequences(
    train_data, maxlen=MAX_LEN, padding="post", truncating="post"
)

test_data = tf.keras.utils.pad_sequences(
    test_data, maxlen=MAX_LEN, padding="post", truncating="post"
)

print("Forma de train_data:", train_data.shape)
print("Forma de test_data:", test_data.shape)


## 3. ¿Qué significa tokenizar?

Una máquina no procesa directamente palabras como `"excellent"` o `"boring"`. Primero se asigna un número a cada palabra del vocabulario. De esta manera, una frase se representa como una secuencia de enteros.

Ejemplo conceptual:

`"excellent movie"` → `[245, 18]`

La secuencia numérica todavía **no es el embedding**. Es la entrada que posteriormente recibe la capa `Embedding`.


In [ ]:
# 4. INSPECCIÓN DEL VOCABULARIO
word_index = tf.keras.datasets.imdb.get_word_index()

# Algunas palabras y sus índices
for palabra in ["excellent", "movie", "boring", "good"]:
    print(palabra, "->", word_index.get(palabra, "no encontrada"))


## 4. Capa de Embedding

La capa `Embedding` aprende un vector para cada token del vocabulario.

Si usamos `EMBEDDING_DIM = 32`, cada palabra queda representada mediante un vector de 32 números:

`token → [x1, x2, x3, ..., x32]`

Durante el entrenamiento, estos vectores se ajustan para ayudar al modelo a resolver la tarea de clasificación.


In [ ]:
# 5. MODELO CON EMBEDDINGS
EMBEDDING_DIM = 32

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=True,
        name="embedding"
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 5. Entrenamiento

Se divide una parte de los datos de entrenamiento para validación. La métrica principal será `accuracy`, mientras que `loss` permite observar el error del modelo.


In [ ]:
# 6. ENTRENAMIENTO
history = model.fit(
    train_data,
    train_labels,
    validation_split=0.2,
    epochs=5,
    batch_size=128,
    verbose=1
)


In [ ]:
# 7. EVALUACIÓN
test_loss, test_accuracy = model.evaluate(test_data, test_labels, verbose=0)

print(f"Pérdida en prueba: {test_loss:.4f}")
print(f"Precisión en prueba: {test_accuracy:.4f}")
print(f"Precisión porcentual: {test_accuracy*100:.2f}%")


## 6. Visualización de resultados

Las gráficas permiten comprobar si el modelo aprende y si existe una diferencia importante entre entrenamiento y validación.


In [ ]:
# 8. GRÁFICA DE PRECISIÓN
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="Entrenamiento")
plt.plot(history.history["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Precisión")
plt.title("Precisión durante el entrenamiento")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 9. GRÁFICA DE PÉRDIDA
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Entrenamiento")
plt.plot(history.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Pérdida durante el entrenamiento")
plt.legend()
plt.grid(True)
plt.show()


## 7. Inspección de los embeddings

Después del entrenamiento podemos acceder a los pesos de la capa `Embedding`. Cada fila corresponde al vector aprendido para un token.


In [ ]:
# 10. EXTRAER MATRIZ DE EMBEDDINGS
embedding_layer = model.get_layer("embedding")
embedding_matrix = embedding_layer.get_weights()[0]

print("Forma de la matriz de embeddings:", embedding_matrix.shape)
print("Vector del token 10:", embedding_matrix[10])


In [ ]:
# 11. MAGNITUD DE ALGUNOS VECTORES
normas = np.linalg.norm(embedding_matrix, axis=1)

print("Norma mínima:", normas.min())
print("Norma máxima:", normas.max())
print("Norma promedio:", normas.mean())


## 8. Predicción de nuevas reseñas

Para probar el modelo con texto nuevo se debe utilizar el mismo vocabulario y la misma transformación que se utilizó durante el entrenamiento.


In [ ]:
# 12. FUNCIONES PARA CONVERTIR TEXTO A SECUENCIA
reverse_word_index = {value: key for key, value in word_index.items()}

def texto_a_secuencia(texto):
    palabras = texto.lower().split()
    secuencia = []
    for palabra in palabras:
        # Se eliminan signos básicos para el ejemplo
        palabra = palabra.strip(".,!?;:'\"()[]")
        indice = word_index.get(palabra)
        if indice is None or indice >= VOCAB_SIZE:
            indice = 2  # token <UNK>
        secuencia.append(indice)
    return tf.keras.utils.pad_sequences(
        [secuencia], maxlen=MAX_LEN, padding="post", truncating="post"
    )

def predecir(texto):
    secuencia = texto_a_secuencia(texto)
    probabilidad = float(model.predict(secuencia, verbose=0)[0][0])
    etiqueta = "positivo" if probabilidad >= 0.5 else "negativo"
    return etiqueta, probabilidad


In [ ]:
# 13. PRUEBAS
ejemplos = [
    "excellent movie good acting",
    "boring bad movie waste time",
    "good story excellent",
    "bad boring terrible"
]

for texto in ejemplos:
    etiqueta, probabilidad = predecir(texto)
    print(f"Texto: {texto}")
    print(f"Predicción: {etiqueta} | Probabilidad positiva: {probabilidad:.3f}")
    print("-" * 60)


## 9. Análisis de resultados

Al finalizar, complete este apartado con los valores que aparezcan al ejecutar el notebook.

**Precisión final en prueba:** ______ %

**Pérdida final en prueba:** ______

**Interpretación sugerida:**

El modelo utiliza una capa de embeddings para convertir los índices de las palabras en vectores densos de 32 dimensiones. Estos vectores se ajustan durante el entrenamiento para que la red pueda encontrar patrones relacionados con el sentimiento de las reseñas. La precisión obtenida en el conjunto de prueba permite estimar qué tan bien generaliza el modelo sobre datos que no utilizó directamente para aprender.

Las curvas de entrenamiento y validación deben analizarse juntas. Si ambas mejoran de forma similar, el aprendizaje es más estable. Si la precisión de entrenamiento continúa aumentando mientras la validación se estanca o disminuye, puede existir sobreajuste.

En las predicciones individuales también es importante recordar que una clasificación no significa que el modelo "entienda" una reseña como lo haría una persona. El modelo identifica patrones estadísticos aprendidos de los datos.


## 10. Conclusiones

1. El texto puede transformarse en una representación numérica mediante tokenización y secuencias de índices.
2. La capa `Embedding` transforma esos índices en vectores densos que se ajustan durante el entrenamiento.
3. Los embeddings permiten alimentar modelos de aprendizaje automático con información textual.
4. El rendimiento debe analizarse mediante métricas y gráficas, no solamente observando una predicción.
5. El experimento demuestra una aplicación práctica de TensorFlow al procesamiento de lenguaje natural.
